## Exercise 6: Shared utility functions, data catalogs

Skills: 
* Import shared utils
* Data catalog
* Use functions to repeat certain data cleaning steps

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_tools/data_catalogs.html

In [ ]:
import geopandas as gpd
import intake
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np

from calitp.tables import tbl
from calitp import query_sql
from siuba import *


import shared_utils
import altair as alt
from shared_utils import altair_utils 
from shared_utils import styleguide

import branca
import folium

#commas to separate 1000s 
pd.options.display.float_format = '{:,}'.format
pd.set_option('display.max_colwidth', None)

## Create a data catalog 

* Include one geospatial data source and one tabular (they should be related...your analysis depends on combining them)
* Import your datasets using the catalog method


#### Opening up geojson

In [ ]:
catalog = intake.open_catalog("./Catalog.yml")

In [ ]:
#Importing geojson 
geojson = catalog.geojson_districts.read()

In [ ]:
geojson = geojson[['DISTRICT','Shape_Length','Shape_Area','geometry']]

In [ ]:
#Keep getting errors...unicode not detected, parquet doesn't work, excel doesn't work, csv won't open properly.
#tircp = catalog.TIRCP_processed.read()

In [ ]:
geojson

#### Open up TIRCP
* Keep getting this weird message: "E0307 18:09:11.701042022    1002 fork_posix.cc:70]           Fork support is only compatible with the epoll1 and poll polling strategies"

In [ ]:
tircp = pd.read_excel('gs://calitp-analytics-data/data-analyses/tircp/tableau_with_temporary_expenditure_sol.xlsx')

In [ ]:
tircp.shape

In [ ]:
#only keep columns I want. Drop NA for district, filter out projects with districts coded as "various"
tircp = (tircp[['Award_Year', 'Grant_Recipient','District','TIRCP_Amount', 
                'County','Project_Title', 'Expended_Amount','Progress','Project_Category']]
         .loc[tircp['District'] != 'Various'].dropna(subset=['District']))

In [ ]:
tircp.shape

In [ ]:
tircp.head(1)

In [ ]:
tircp.Project_Category.value_counts()

In [ ]:
  def percentiles(row):
            if row.Project_Category == 'Medium':
                return 50
            elif row.Project_Category == 'Large':
                return 75
            else:
                return 25

In [ ]:
tircp["Percentiles"] =tircp.apply(lambda x: percentiles(x), axis=1)
    

## Combine datasets 

Tiffany's Notes
* Do a merge or spatial join to combine the geospatial and tabular data
* Create a new column of a summary statistic to visualize
* Rely on `shared_utils` to do at least one operation (aggregation, re-projecting to a different CRS, exporting geoparquet, etc)


* Do left join on geojson, code the missing districts with dummy values so the districts will show up.

In [ ]:
#Joining the 2 dataframes
joined = geojson.merge(tircp, how = 'outer', left_on ='DISTRICT', right_on = 'District', indicator = True)

In [ ]:
joined._merge.value_counts()

In [ ]:
#replacing any districts with 0 projects with 0. 
#joined = joined.replace(np.nan, 0)

In [ ]:
#Get rid of unneeded columns
joined = joined[['Shape_Length', 'Shape_Area', 'geometry', 'Award_Year',
       'Grant_Recipient', 'DISTRICT', 'TIRCP_Amount', 'County', 'Project_Title','Expended_Amount', 'Progress', 'Project_Category', 'Percentiles'
       ]] 

### Aggregating using a function from shared_utils

In [ ]:
aggregation_function = shared_utils.geography_utils.aggregate_by_geography(joined, ['DISTRICT'], 
                                                                           sum_cols = ['TIRCP_Amount', 'Expended_Amount'],
                                                                           mean_cols = ['Percentiles'], 
                                                                           nunique_cols =['Project_Title'])


In [ ]:
#Replace the districts with 0 projects with the appropriate strings.
aggregation_function.loc[(aggregation_function['TIRCP_Amount'] == 0), "Project_Title"] = 0

In [ ]:
aggregation_function = aggregation_function.dropna(how = 'all') 

In [ ]:
aggregation_function.sort_values('TIRCP_Amount')

### Looking at TIRCP received by district - keeping geometry.
* https://matplotlib.org/2.0.2/users/colormaps.html

In [ ]:
#aggregated df to keep geometry.
subset_TIRCP = joined[['DISTRICT', 'geometry','TIRCP_Amount']]
total_TIRCP_2 = subset_TIRCP.dissolve(by='DISTRICT', aggfunc='sum').sort_values('TIRCP_Amount').rename(columns = {'TIRCP_Amount':'Total_TIRCP_Received'}).reset_index()

In [ ]:
#aggregated df to keep geometry.
subset_percentiles = joined[['DISTRICT', 'geometry','Percentiles']]
total_percentiles2 = subset_percentiles.dissolve(by='DISTRICT', aggfunc='mean').sort_values('Percentiles').reset_index()

In [ ]:
joined_map = (pd.merge(total_percentiles2, total_TIRCP_2,  on=['DISTRICT'])
.drop(columns = ['geometry_x']).rename(columns = {'geometry_y':'geometry'}))

In [ ]:
joined_map.loc[(joined_map['DISTRICT'] == 9), "Percentiles"] = 25.0

In [ ]:
joined_map[['DISTRICT','Percentiles','Total_TIRCP_Received']]

### Looking at # of Projects in a District -- keeping geometry

In [ ]:
#aggregated df to keep geometry for # of projects
subset_projects = joined[['DISTRICT', 'geometry','Project_Title']]
total_projects_2 = subset_projects.dissolve(by='DISTRICT', aggfunc='nunique').sort_values('DISTRICT').rename(columns = {'Project_Title':'Total_Projects'}).reset_index()

### Ipyleaflet Map

In [ ]:
REGION_CENTROIDS = shared_utils.map_utils.REGION_CENTROIDS

In [ ]:
colorscale_TIRCP = branca.colormap.StepColormap(
                colors=["#F6BF16", "#E16B26",  "#00896B"], 
                index=[0, 93_614_000, 351_601_500], 
                vmin=0, vmax=2_458_538_000,
)
colorscale_TIRCP

In [ ]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(2458538000),
            "plot_col_name": 'Total_TIRCP_Received',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [ ]:

shared_utils.map_utils.make_ipyleaflet_choropleth_map(joined_map, 
                                                     plot_col = 'Total_TIRCP_Received',
                                                     geometry_col = 'DISTRICT', 
                                                     choropleth_dict = choropleth_dict, 
                                                     colorscale = colorscale_TIRCP,
                                                     zoom=5, 
                                                     centroid = REGION_CENTROIDS['CA'][0])

### Folium Map  


In [ ]:
TOOLTIP_KWARGS = {
    "min_width": 50,
    "max_width": 100,
    "font_size": "12px",
}


In [ ]:
popup_dict = {"DISTRICT": "Caltrans District",
    
    "Total_TIRCP_Received": "Total TIRCP Funding Received" 
}

In [ ]:
tooltip_dict = {"DISTRICT": "Caltrans District",
    
    "Total_TIRCP_Received":"Total TIRCP Funding Received"  }

In [ ]:
color_scale_percentiles = branca.colormap.StepColormap(
                colors=["#F6BF16", "#E16B26",  "#00896B"], 
                index=[0, 25, 50], 
                vmin=0, vmax=100,
)
color_scale_percentiles

In [ ]:
joined_map[['DISTRICT','Percentiles']]

In [ ]:
shared_utils.map_utils.make_folium_choropleth_map(joined_map,
    plot_col = 'Percentiles',
    popup_dict = popup_dict,
    tooltip_dict = tooltip_dict,
    colorscale = color_scale_percentiles,
    fig_width = int(700),
    fig_height = int(700),
    zoom=REGION_CENTROIDS["CA"][1],
    centroid=REGION_CENTROIDS["CA"][0],
    title="TIRCP Funds Received by District",
    legend_name="DISTRICT",)

### Making a map without the function & add Markers

* https://towardsdatascience.com/using-folium-to-generate-choropleth-map-with-customised-tooltips-12e4cec42af2
* https://geopandas.org/en/stable/gallery/plotting_with_folium.html

In [ ]:
#Bins for legend.
bins = list(total_TIRCP_2["Total_TIRCP_Received"].quantile([0, 0.25, 0.5, 0.75, 1]))
bins

In [ ]:
TIRCP_centroid = total_TIRCP_2.copy()

In [ ]:
TIRCP_centroid['Centroid'] = TIRCP_centroid.geometry.centroid

In [ ]:
TIRCP_centroid = TIRCP_centroid.drop(columns=['geometry'])

In [ ]:
type(TIRCP_centroid)

In [ ]:
TIRCP_centroid

In [ ]:
geo_df_list = [[point.xy[1][0], point.xy[0][0]] for point in TIRCP_centroid.Centroid ]

In [ ]:
geo_df_list

In [ ]:
i = 0
for coordinates in geo_df_list:
    #assign a color marker for the type of volcano, Strato being the most common
    if TIRCP_centroid.DISTRICT[i] == 1:
        type_color = "green"
    elif TIRCP_centroid.DISTRICT[i] == 2:
        type_color = "blue"
    elif TIRCP_centroid.DISTRICT[i] == 3:
        type_color = "orange"
    elif TIRCP_centroid.DISTRICT[i] == 4:
        type_color = "pink"
    elif TIRCP_centroid.DISTRICT[i] == 5:
        type_color = "red"
    elif TIRCP_centroid.DISTRICT[i] == 6:
        type_color = "yellow"
    else:
        type_color = "purple"


In [ ]:
#Base map
TIRCP_map = folium.Map(location=[35.8, -119.4], zoom_start=5,
                       width = 500, height=500)

In [ ]:
#Add custom properties to the map
folium.Choropleth(
    geo_data = total_TIRCP_2,#geo dataframe with our geometry
    name ='choropleth',                  
    data = total_TIRCP_2,       #the data we want to graph.              
    columns = ['DISTRICT', 'Total_TIRCP_Received'], #Colors on map is based on TIRCP, DISTRICT Is key of data to map with location
    key_on ='feature.properties.DISTRICT', #the geography we want to graph: district in this case.
    fill_color ='YlGnBu',     
    fill_opacity = 0.7,
    line_opacity = 0.2,
    legend_name = "DISTRICT", 
    bins = bins, #customizing legend with bins made from percentiles
    highlight = True
).add_to(TIRCP_map)


In [ ]:
TIRCP_map.add_child(folium.Marker(location = coordinates,
                            popup =
                            "District: " + str(TIRCP_centroid.DISTRICT[i]) + '<br>' +
                            
                            "TIRCP Recieved: " + str(TIRCP_centroid.Total_TIRCP_Received[i]) + '<br>'
                            "Coordinates: " + str(geo_df_list[i]),
                            icon = folium.Icon(color = "%s" % type_color)))
i = i + 1

In [ ]:
TIRCP_map

## Use functions to do parameterized visualizations
* Use a function to create your chart
* Within the function, the colors should use the Cal-ITP theme that is available in `styleguide`
* Within the function, there should be at least 1 parameter that changes (ex: chart title reflects the correct county, legend title reflects the correct county, etc)
* Produce 3 charts, using your function each time, and have the function correctly insert the parameters 

In [ ]:
#Change back to names
aggregation_function['DISTRICT'] = (aggregation_function['DISTRICT'].replace({7:'District 7: Los Angeles',
                                            4:'District 4: Bay Area / Oakland',
                                            2: 'District 2: Redding',
                                            9: 'District 9: Bishop',
                                            'VAR':'Various',
                                            10:'District 10: Stockton',
                                            11:'District 11: San Diego',
                                            3:'District 3: Marysville / Sacramento',
                                            12: 'District 12: Orange County',
                                            8: 'District 8: San Bernardino / Riverside',
                                            5:'District 5: San Luis Obispo / Santa Barbara',
                                            6:'District 6: Fresno / Bakersfield',
                                            1:'District 1: Eureka'}))
    

In [ ]:
aggregation_function = aggregation_function.rename(columns ={'Project_Title':'Number_of_Projects', 'TIRCP_Amount':'Total_TIRCP_Received', 
                                      'DISTRICT':'District'}) 

In [ ]:
#Round monetary cols into millions for more readabilty
aggregation_function['TIRCP_in_millions'] =  (aggregation_function['Total_TIRCP_Received'].astype(float)/1000000).round()
aggregation_function['Expended_in_millions'] = (aggregation_function['Expended_Amount'].astype(float)/1000000).round()
aggregation_function['TIRCP_in_millions_labels'] = '$' + (aggregation_function['Total_TIRCP_Received'].astype(float)/1000000).round().astype(str) 
aggregation_function['Expended_in_millions_labels'] ='$' + (aggregation_function['Expended_Amount'].astype(float)/1000000).round().astype(str) 
aggregation_function

In [ ]:
#Function for cleaning up X and Y column names.
def labeling(word):
    # Add specific use cases where it's not just first letter capitalized
    LABEL_DICT = { "prepared_y": "Year",
              "dist": "District",
              "nunique":"Number of Unique",
              "project_no": "Project Number"}
    
    if (word == "mpo") or (word == "rtpa"):
        word = word.upper()
    elif word in LABEL_DICT.keys():
        word = LABEL_DICT[word]
    else:
        #word = word.replace('n_', 'Number of ').title()
        word = word.replace('unique_', "Number of Unique ").title()
        word = word.replace('_', ' ').title()
    
    return word


In [ ]:
# Sort legend by district #. 
DISTRICTS = ['District 1: Eureka',
             'District 2: Redding',
             'District 3: Marysville / Sacramento',
             'District 4: Bay Area / Oakland', 
             'District 5: San Luis Obispo / Santa Barbara',
             'District 6: Fresno / Bakersfield',
             'District 7: Los Angeles',
             'District 8: San Bernardino / Riverside',
             'District 9: Bishop',
             'District 10: Stockton',
             'District 11: San Diego',
             'District 12: Orange County']

In [ ]:
#Base bar chart
def base_bar(df):
    chart = (alt.Chart(df)
             .mark_bar()
             )
    return chart

#Function
def make_bar(df, y_col, x_col, label_col, chart_title=''):
    
    if chart_title == "":
        chart_title = (f"{labeling(x_col)} by {labeling(y_col)}")
    
    bar = base_bar(df)
    
    bar = (bar.encode(
         x=alt.X(x_col, title=labeling(x_col)),
         y=alt.Y(y_col, title=labeling(y_col), sort=('-x')),
         color=alt.Color(y_col, 
                        scale=alt.Scale(
                            domain=DISTRICTS, #Specifies the order of the legend.
                            range=altair_utils.CALITP_CATEGORY_BRIGHT_COLORS
                        )
                )
             )
            )
    #https://stackoverflow.com/questions/54015250/altair-setting-constant-label-color-for-bar-chart
    text = (bar
            .mark_text(align="left", baseline="middle",
                       color="black", dy=3
                      )
            .encode(text=label_col, 
                    # Set color here, because encoding for mark_text gets 
                    # superseded by alt.Color
                   color=alt.value("black"))
    )
      
    chart = (bar+text)
    
    chart = (styleguide.preset_chart_config(chart)
             .properties(title= chart_title).configure_axis(grid=False)
            )
    
    display(chart)

In [ ]:
make_bar(aggregation_function, 'District', 'Number_of_Projects', 
         'Number_of_Projects', 'Total Projects in each District')

In [ ]:
make_bar(aggregation_function, 'District', 'TIRCP_in_millions', 
         'TIRCP_in_millions_labels', 'Total TIRCP Received (in millions) by District')

In [ ]:
make_bar(aggregation_function, 'District', 'Expended_in_millions', 
         'Expended_in_millions_labels', 'Expended Amounts (in millions) by District')